
# 04 | Team Performance Analysis

**Objective:** Analyze attacking and defensive performance across the 2026 FIFA World Cup dataset using SQL.

I compare national teams using match results, scoring efficiency, defensive statistics, and tournament performance to identify meaningful patterns and differences.



## 4.1 | Connecting to the Database

I connect to my SQLite database to analyze team performance using the validated match and team datasets.


In [1]:

import pandas as pd
import sqlite3
from pathlib import Path

# Connect to existing database
db_path = Path.cwd().parent / "Data" / "Prepared" / "world_cup_2026.db"

conn = sqlite3.connect(db_path)

print("Connected to World Cup database!")


Connected to World Cup database!



## 4.2 | Evaluating Defensive Performance

I compare teams based on goals conceded, clean sheets, and defensive averages to identify the strongest defensive records in the tournament.


In [2]:

# Evaluate defensive performance across teams
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        away_score AS goals_conceded
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        home_score AS goals_conceded
    FROM matches
)

SELECT
    team,
    COUNT(*) AS matches_played,
    SUM(goals_conceded) AS total_conceded,
    ROUND(AVG(goals_conceded), 2) AS conceded_per_match,
    SUM(CASE
        WHEN goals_conceded = 0 THEN 1
        ELSE 0
    END) AS clean_sheets,
    ROUND(
        100.0 * SUM(CASE
            WHEN goals_conceded = 0 THEN 1
            ELSE 0
        END) / COUNT(*), 1
    ) AS clean_sheet_pct
FROM team_matches
GROUP BY team
HAVING COUNT(*) >= 4
ORDER BY conceded_per_match ASC, clean_sheet_pct DESC;
"""

defensive_df = pd.read_sql_query(query, conn)

display(defensive_df.head(15))


,team,matches_played,total_conceded,conceded_per_match,clean_sheets,clean_sheet_pct
0,Spain,8,1,0.13,7,87.5
1,Colombia,5,1,0.20,4,80.0
2,Mexico,5,3,0.60,4,80.0
3,Portugal,5,3,0.60,2,40.0
4,Australia,4,3,0.75,2,50.0
5,Ghana,4,3,0.75,2,50.0
6,Brazil,5,4,0.80,2,40.0
7,Côte d'Ivoire,4,4,1.00,2,50.0
8,Morocco,6,6,1.00,2,33.3
9,Switzerland,6,6,1.00,2,33.3



## 4.3 | Evaluating Attacking Performance

I evaluate attacking performance by comparing total goals, scoring averages, and the frequency with which teams score in their matches.


In [3]:

# Evaluate attacking performance across teams
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        home_score AS goals_scored
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        away_score AS goals_scored
    FROM matches
)

SELECT
    team,
    COUNT(*) AS matches_played,
    SUM(goals_scored) AS total_goals,
    ROUND(AVG(goals_scored), 2) AS goals_per_match,

    SUM(CASE
        WHEN goals_scored > 0 THEN 1
        ELSE 0
    END) AS matches_scored,

    SUM(CASE
        WHEN goals_scored = 0 THEN 1
        ELSE 0
    END) AS scoreless_matches,

    ROUND(
        100.0 * SUM(CASE
            WHEN goals_scored > 0 THEN 1
            ELSE 0
        END) / COUNT(*), 1
    ) AS scoring_pct

FROM team_matches
GROUP BY team
HAVING COUNT(*) >= 4
ORDER BY goals_per_match DESC, scoring_pct DESC;
"""

attacking_df = pd.read_sql_query(query, conn)

display(attacking_df.head(15))


,team,matches_played,total_goals,goals_per_match,matches_scored,scoreless_matches,scoring_pct
0,Germany,4,11,2.75,4,0,100.0
1,Netherlands,4,11,2.75,4,0,100.0
2,Senegal,4,10,2.50,4,0,100.0
3,England,8,20,2.50,7,1,87.5
4,France,8,20,2.50,7,1,87.5
5,Argentina,8,19,2.38,7,1,87.5
6,Belgium,6,14,2.33,5,1,83.3
7,USA,5,11,2.20,5,0,100.0
8,Norway,6,13,2.17,6,0,100.0
9,Brazil,5,10,2.00,5,0,100.0



## 4.4 | Comparing Overall Team Performance

I combine attacking and defensive statistics to compare scoring efficiency, defensive strength, and goal difference across national teams.


In [4]:

# Combine attacking and defensive statistics using SQL
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        home_score AS goals_for,
        away_score AS goals_against
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        away_score AS goals_for,
        home_score AS goals_against
    FROM matches
),

attacking AS (
    SELECT
        team,
        COUNT(*) AS matches_played,
        SUM(goals_for) AS goals_scored,
        ROUND(AVG(goals_for), 2) AS goals_per_match
    FROM team_matches
    GROUP BY team
),

defensive AS (
    SELECT
        team,
        SUM(goals_against) AS goals_conceded,
        ROUND(AVG(goals_against), 2) AS conceded_per_match,
        SUM(CASE WHEN goals_against = 0 THEN 1 ELSE 0 END) AS clean_sheets
    FROM team_matches
    GROUP BY team
)

SELECT
    a.team,
    a.matches_played,
    a.goals_scored,
    a.goals_per_match,
    d.goals_conceded,
    d.conceded_per_match,
    d.clean_sheets,
    a.goals_scored - d.goals_conceded AS goal_difference
FROM attacking a
JOIN defensive d ON a.team = d.team
WHERE a.matches_played >= 4
ORDER BY goal_difference DESC;
"""

performance_df = pd.read_sql_query(query, conn)

display(performance_df.head(15))


,team,matches_played,goals_scored,goals_per_match,goals_conceded,conceded_per_match,clean_sheets,goal_difference
0,Spain,8,14,1.75,1,0.13,7,13
1,Argentina,8,19,2.38,8,1.00,2,11
2,France,8,20,2.50,10,1.25,4,10
3,England,8,20,2.50,12,1.50,2,8
4,Belgium,6,14,2.33,7,1.17,1,7
5,Mexico,5,10,2.00,3,0.60,4,7
6,Brazil,5,10,2.00,4,0.80,2,6
7,Germany,4,11,2.75,5,1.25,0,6
8,Netherlands,4,11,2.75,5,1.25,0,6
9,Portugal,5,8,1.60,3,0.60,2,5



## 4.5 | Analyzing Performance by Tournament Stage

I compare Spain and Argentina across the group stage and knockout rounds to evaluate how their attacking and defensive performance changed throughout the tournament.


In [5]:

# Compare performance by tournament stage
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        "group" AS tournament_stage,
        home_score AS goals_for,
        away_score AS goals_against
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        "group" AS tournament_stage,
        away_score AS goals_for,
        home_score AS goals_against
    FROM matches
),

stage_performance AS (
    SELECT
        team,
        CASE
            WHEN tournament_stage LIKE 'Group%'
            THEN 'Group Stage'
            ELSE 'Knockout Stage'
        END AS stage,
        goals_for,
        goals_against
    FROM team_matches
    WHERE team IN ('Spain', 'Argentina')
)

SELECT
    team,
    stage,
    COUNT(*) AS matches_played,
    SUM(goals_for) AS goals_scored,
    SUM(goals_against) AS goals_conceded,
    ROUND(AVG(goals_for), 2) AS goals_per_match,
    ROUND(AVG(goals_against), 2) AS conceded_per_match,
    SUM(goals_for - goals_against) AS goal_difference
FROM stage_performance
GROUP BY team, stage
ORDER BY team, stage;
"""

stage_df = pd.read_sql_query(query, conn)

display(stage_df)


,team,stage,matches_played,goals_scored,goals_conceded,goals_per_match,conceded_per_match,goal_difference
0,Argentina,Group Stage,3,8,1,2.67,0.33,7
1,Argentina,Knockout Stage,5,11,7,2.20,1.40,4
2,Spain,Group Stage,3,5,0,1.67,0.00,5
3,Spain,Knockout Stage,5,9,1,1.80,0.20,8



## 4.6 | Tracking Cumulative Team Performance

I track Spain and Argentina's cumulative goals scored, goals conceded, and goal difference to understand how their performance developed throughout the tournament.


In [6]:

# Calculate cumulative performance by match
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        match_number,
        "group" AS stage,
        home_score AS goals_for,
        away_score AS goals_against
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        match_number,
        "group" AS stage,
        away_score AS goals_for,
        home_score AS goals_against
    FROM matches
)

SELECT
    team,
    match_number,
    stage,
    goals_for,
    goals_against,

    SUM(goals_for) OVER (
        PARTITION BY team
        ORDER BY match_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_goals,

    SUM(goals_against) OVER (
        PARTITION BY team
        ORDER BY match_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_conceded,

    SUM(goals_for - goals_against) OVER (
        PARTITION BY team
        ORDER BY match_number
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_goal_difference

FROM team_matches
WHERE team IN ('Spain', 'Argentina')
ORDER BY team, match_number;
"""

cumulative_df = pd.read_sql_query(query, conn)

display(cumulative_df)


,team,match_number,stage,goals_for,goals_against,cumulative_goals,cumulative_conceded,cumulative_goal_difference
0,Argentina,19,Group J,3,0,3,0,3
1,Argentina,43,Group J,2,0,5,0,5
2,Argentina,70,Group J,3,1,8,1,7
3,Argentina,86,Round of 32,3,2,11,3,8
4,Argentina,95,Round of 16,3,2,14,5,9
5,Argentina,100,Quarter-ﬁnal,3,1,17,6,11
6,Argentina,102,Semi-ﬁnal,2,1,19,7,12
7,Argentina,104,Final,0,1,19,8,11
8,Spain,14,Group H,0,0,0,0,0
9,Spain,38,Group H,4,0,4,0,4



## 4.7 | Evaluating Match-to-Match Consistency

I use SQL window functions to compare consecutive match results and identify changes in scoring performance throughout the tournament.


In [7]:

# Compare scoring performance between consecutive matches
query = """
WITH team_matches AS (
    SELECT
        home_team AS team,
        match_number,
        home_score AS goals_scored,
        away_score AS goals_conceded
    FROM matches

    UNION ALL

    SELECT
        away_team AS team,
        match_number,
        away_score AS goals_scored,
        home_score AS goals_conceded
    FROM matches
),

match_progression AS (
    SELECT
        team,
        match_number,
        goals_scored,
        goals_conceded,

        LAG(goals_scored) OVER (
            PARTITION BY team
            ORDER BY match_number
        ) AS previous_goals

    FROM team_matches
    WHERE team IN ('Spain', 'Argentina')
)

SELECT
    team,
    match_number,
    goals_scored,
    goals_conceded,
    previous_goals,
    goals_scored - previous_goals AS scoring_change

FROM match_progression
ORDER BY team, match_number;
"""

consistency_df = pd.read_sql_query(query, conn)

display(consistency_df)


,team,match_number,goals_scored,goals_conceded,previous_goals,scoring_change
0,Argentina,19,3,0,NaN,NaN
1,Argentina,43,2,0,3.0,-1.0
2,Argentina,70,3,1,2.0,1.0
3,Argentina,86,3,2,3.0,0.0
4,Argentina,95,3,2,3.0,0.0
5,Argentina,100,3,1,3.0,0.0
6,Argentina,102,2,1,3.0,-1.0
7,Argentina,104,0,1,2.0,-2.0
8,Spain,14,0,0,NaN,NaN
9,Spain,38,4,0,0.0,4.0



## 4.8 | Key Findings

My analysis highlights several differences in team performance across the tournament:

- **Spain:** Finished with the highest goal difference (+13) and conceded only one goal across eight matches.
- **Argentina:** Scored more goals than Spain (19 vs. 14), but also conceded significantly more (8 vs. 1).
- **France and England:** Led the tournament in total goals scored, with 20 each.
- **Defensive performance:** Spain maintained seven clean sheets, demonstrating exceptional defensive consistency.

Using SQL, I compared attacking and defensive performance, analyzed tournament stages, and tracked match-by-match trends with window functions.

These findings are based on the imported dataset and should be interpreted within its documented source limitations.
